# 00. AxonML Overview

In this tutorial, we provide a brief overview of AxonML. We will cover the following topics:

- Setting up a neuronal model
- Accessing and viewing model properties
- Running simulations
- Analyzing simulation results


## What is AxonML?

AxonML is a Python library designed for simulating and analyzing neuronal models. It provides a flexible framework for building complex neuronal populations and networks, allowing researchers to explore the dynamics of neural systems.


## What are "neuronal models", conceptually?

Neuronal models are mathematical representations of neurons, their responses to intra- and extracellular stimulation, and their interactions within a network. They can range from simple point neuron models to complex multi-compartmental models that capture the detailed morphology and biophysics of real neurons. Neuronal models are used to study various aspects of neural function, including the behavior of cellular substrates in neural interfaces, signal propagation, synaptic integration, and network dynamics.

In AxonML, the core abstraction is the {py:class}`~axonml.models.core.Population`, which represents a group of neurons with shared properties. Each {py:class}`~axonml.models.core.Population` can contain multiple compartments, each with its own biophysical properties and dynamics.

Fundamentally, a {py:class}`~axonml.models.core.Population` in AxonML is an array of compartments. These compartments may be connected topologically to form neurons, but the {py:class}`~axonml.models.core.Population` itself is agnostic to this structure. Each compartment can have its own set of properties, such as membrane potential, ion channel densities, and synaptic inputs. When the compartments of a {py:class}`~axonml.models.core.Population` are connected (there exist finite-impedance current paths between them), a row of the {py:class}`~axonml.models.core.Population` corresponds to a single neuron, and a column corresponds to a compartment within that neuron. When compartments are not connected, each compartment can be treated as an independent point neuron.

AxonML implements a number of abstractions on top of {py:class}`~axonml.models.core.Population` to facilitate working with neuronal models:
- {py:class}`~axonml.models.core.Population` – generic multicompartment neuron population (`N` neurons × `C` compartments)
  - {py:class}`~axonml.models.core.SingleCompartment` ({py:class}`~axonml.models.core.Population`) – single‑compartment (point‑neuron) population (`N` × `C` total neurons)
  - {py:class}`~axonml.models.core.Axon` ({py:class}`~axonml.models.core.Population`) – 1D cable‑like axonal fiber population
    - {py:class}`~axonml.models.core.Unmyelinated` ({py:class}`~axonml.models.core.Axon`) – unmyelinated axons with uniform spatial discretization
    - {py:class}`~axonml.models.core.Myelinated` ({py:class}`~axonml.models.core.Axon`) – myelinated axons with nodes of Ranvier and diameter‑scaled internodes
    - {py:class}`~axonml.models.extcell.ExtCellAxon` ({py:class}`~axonml.models.core.Axon`) – axons with two‑layer extracellular coupling (e.g., realistic myelin)
  - {py:class}`~axonml.models.tree.Tree` ({py:class}`~axonml.models.core.Population`) – branched, tree‑shaped neuron morphologies (soma, axon, dendrites, apical dendrites)
    - {py:class}`~axonml.models.extcell.ExtCellTree` ({py:class}`~axonml.models.tree.Tree`) – tree‑shaped neurons with two‑layer extracellular coupling


Each of these abstractions builds upon the {py:class}`~axonml.models.core.Population` class, adding specific features and behaviors relevant to the type of neuronal model being represented. This hierarchical structure allows users to easily create and manipulate different types of neuronal models while leveraging the common functionality provided by the {py:class}`~axonml.models.core.Population` base class.

## Setting Up a Neuronal Model

To set up a neuronal model in AxonML, you typically start by creating an instance of a {py:class}`~axonml.models.core.Population` or one of its subclasses. You then define its biohysical properties. Lets look at an example of creating a simple unmyelinated axon model:

In [ ]:
import axonml as ax
from axonml.units import um, mm

axon = ax.Unmyelinated(diameters=[1.0 * um], dx=10.0 * um, L=2.0 * mm, v_init=-65.0, celsius=6.3)

This defines an unmyelinated axon with a uniform diameter of 1 micrometer (note how diameters is a list length 1, if we had supplied multiple diameters we have multiple axons), compartment length of 10 micrometers, total length 2.0 millimeters, an initial membrane potential of -65 mV, and a temperature of 6.3 degrees Celsius. For unmyelinated axon models, AxonML will round up the number of compartments to ensure an odd number, which ensures an unambiguous central compartment for stimulation. By default, axons lie along the x-axis, with the central compartment at x=0.

In [ ]:
print(axon.L, axon.nc, axon.x[0, int(axon.nc // 2)], axon.x[0, 0], axon.x[0, -1]) 

This model doesn't do much yet, as we haven't defined any channel biophysics. Let's make it a simple Hodgkin-Huxley axon by adding the standard sodium, potassium, and leak channels:

In [ ]:
from axonml.models.mod import hh

axon.insert(hh)

This inserts the Hodgkin-Huxley channel model into all compartments of the axon, enabling it to exhibit action potential dynamics when stimulated. At this point, we haven't yet built the model, only defined its structure and biophysics. 

```{important}
At this point, once the model structure and biophysics are defined, it is safe to move the model to a specific device (CPU or GPU) using the `.to(device)` method. This ensures that all subsequent computations are performed on the specified device.
```

In the next sections, we will explore how to run simulations and analyze the results. First, we initialize the model to prepare it for simulation:

In [ ]:
axon = axon.initialize()

The model is now initialized and ready for simulation. All state variables have been set to their initial conditions based on the defined biophysics. To help us verify that our model is set up correctly, let's inspect some of its properties. For this, it is helpful to introduce the concept of a {py:class}`~axonml.models.slice.Slice`, which is a central abstraction in AxonML for accessing and manipulating subsets of a {py:class}`~axonml.models.core.Population`.

## Slices

A {py:class}`~axonml.models.slice.Slice` in AxonML is a way to access and manipulate specific subsets of a {py:class}`~axonml.models.core.Population`. It allows you to select particular neurons and compartments, making it easier to work with large populations. You can think of a {py:class}`~axonml.models.slice.Slice` as a view into the {py:class}`~axonml.models.core.Population`, where you can read and modify properties of the selected compartments. Slices can be created using standard Python array indexing syntax:

In [ ]:
# view the membrane potential of compartments 2 to 4 in the first (and only) axon
print(axon[0, 2:5].v)

# view the sodium channel activation variable 'm' in the central compartment of the first axon
print(axon[0, axon.nc // 2].mech.hh.m)

Slices can also be labeled and accessed by name, which can make code more readable when working with complex models. For example, you might label the central compartment of an axon as "node" and access it using that label:

In [ ]:
axon[0, axon.nc // 2].label("node")
print(axon.node.mech.hh.m)

For more complex morphologies (such as {py:class}`~axonml.models.tree.Tree` populations), AxonML implements various quality-of-life features to facilitate working with compartments by name, such as automatic labeling of standard structures (e.g., "soma", "axon", "dendrite", "apical") and convenient access to commonly used subsets (e.g., all dendritic compartments), which we will explore in later tutorials.

## Running a simulation

This model is ready to be simulated, but it is not very interesting yet since we have not applied any stimulation. AxonML allows us to apply intracellular current injetion to specific compartments using the same {py:class}`~axonml.models.slice.Slice` abstraction we used to access model properties. For example, we can inject a 1 nA current pulse into the terminal compartment of the first axon for 0.1 ms starting at t=0.1 ms:

In [ ]:
from axonml.units import nA, ms
stim = ax.mono_rect(amp=1.0 * nA, pw=0.1 * ms, delay=0.1 * ms)
axon[0, 0].inject(stim)

In the above, {py:class}`ax.mono_rect <axonml.models.stim.waveform.implementations.mono_rect>` is an example of a {py:class}`~axonml.models.stim.waveform.core.Waveform` object that defines a simple rectangular current pulse. AxonML provides various built-in waveform types, and users can also define custom waveforms as needed (see the "axonml.models.stim" API documentation for more details). Ok, now we are ready to run the simulation! First though, we need to set up some recording to capture the results. Let's record the membrane potential at the center, 75% along, and terminal compartments of the axon:

In [ ]:
from axonml.models.callbacks import Recorder

rec = Recorder(['v'], node_indices=axon.c(0.5, 0.75, 1.0))

Now lets run and record!

In [ ]:
rec.reset()
axon.initialize(force_rebuild=True)
axon.run(tstop=25.0 * ms, dt=0.01 * ms, callbacks=[rec])

In [ ]:
vm_rec = rec.numpy('v')
print(vm_rec.shape)  # (num_timepoints, num_neurons, num_recorded_compartments)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
t = np.arange(0, 25.0+0.01, 0.01)
plt.plot(t, vm_rec[:, 0, 0], label='Center')
plt.plot(t, vm_rec[:, 0, 1], label='75% along')
plt.plot(t, vm_rec[:, 0, 2], label='Terminal')
plt.xlabel('Time (ms)')
plt.ylabel('Membrane potential (mV)')
plt.legend()
plt.show()

Awesome! We have successfully run a simulation of our unmyelinated axon model with intracellular current injection and recorded the membrane potential at specified compartments. In the next tutorial, we will look at extracellular stimulation and recording.